## **Fine-tune progress — train vs validation error**

Visualise the per-epoch training/validation metrics for a kraken fine-tune run.

Two data sources:
1. **`epoch_stats.json`** in the run directory — written by `finetune.py` at the END of training. Use this once training has completed.
2. **Hand-entered values** from the terminal progress bar — for inspecting an in-progress run before `epoch_stats.json` exists.

The notebook picks whichever is present and plots train loss, validation accuracy, and validation error (1 − val_accuracy) on the same axis.

In [ ]:
import json
import os
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
from dotenv import load_dotenv

load_dotenv()
project_root = Path(os.environ.get("PROJECT_ROOT", "."))

RUN_DIR = project_root / "models/ocr/finetuned/finetune_20260614_133655"
print(f"run dir: {RUN_DIR}")
print(f"epoch_stats.json present: {(RUN_DIR / 'epoch_stats.json').exists()}")

### Snapshot from terminal progress bar (epochs 0–8, manual)

These are the values printed at each epoch end in the progress bar of the still-running training. Add a row below for each new completed epoch until `epoch_stats.json` lands.

In [ ]:
manual_rows = [
    # epoch, train_loss_epoch, val_accuracy, val_word_accuracy
    (0, 72.785, 0.99627, 0.981),
    (1, 25.868, 0.99721, 0.986),
    (2, 20.449, 0.99785, 0.989),
    (3, 19.251, 0.99785, 0.988),
    (4, 19.308, 0.99785, 0.988),
    (5, 20.208, 0.99786, 0.989),
    (6, 21.370, 0.99786, 0.988),
    (7, 23.437, 0.99786, 0.989),
    (8, 23.437, 0.99786, 0.989),
]
manual_df = pd.DataFrame(
    manual_rows,
    columns=["epoch", "train_loss_epoch", "val_accuracy", "val_word_accuracy"],
)

### Load metrics: prefer `epoch_stats.json`, fall back to manual

If training finished and the JSON exists, we use that — same numbers but for any number of epochs. Otherwise we use the manual snapshot above.

In [ ]:
stats_path = RUN_DIR / "epoch_stats.json"

if stats_path.exists():
    stats = json.loads(stats_path.read_text(encoding="utf-8"))
    df = pd.DataFrame(stats)
    source = f"epoch_stats.json ({len(df)} epochs)"
else:
    df = manual_df.copy()
    source = f"manual snapshot ({len(df)} epochs — training still running)"

df["val_error"]      = 1.0 - df["val_accuracy"]
df["val_word_error"] = 1.0 - df["val_word_accuracy"]

print(f"source: {source}")
df

### Train loss vs validation error

Two curves on twin y-axes so you can see both trends despite being on different scales:
- **Left axis** — `train_loss_epoch` (kraken's mean CTC loss across all training batches in the epoch). Lower = better.
- **Right axis** — `val_error = 1 − val_accuracy` (the per-character error rate, i.e. **val_CER**). Lower = better.

Both should decrease over time if the model is still learning. A flattening or upward bend on validation while training loss keeps going down = overfitting.

In [ ]:
fig, ax_l = plt.subplots(figsize=(11, 5))
ax_r = ax_l.twinx()

l1, = ax_l.plot(df["epoch"], df["train_loss_epoch"], marker="o",
                color="#3a4a6a", label="train_loss_epoch")
l2, = ax_r.plot(df["epoch"], df["val_error"], marker="s",
                color="#b53", label="val_error (= 1 − val_accuracy = val_CER)")

ax_l.set_xlabel("epoch")
ax_l.set_ylabel("train_loss_epoch (mean CTC loss)", color="#3a4a6a")
ax_r.set_ylabel("val_error",                       color="#b53")
ax_l.tick_params(axis="y", labelcolor="#3a4a6a")
ax_r.tick_params(axis="y", labelcolor="#b53")

ax_l.set_title("Train loss vs validation error per epoch")
ax_l.grid(True, alpha=0.3)
ax_l.legend([l1, l2], [l1.get_label(), l2.get_label()], loc="upper right")
plt.tight_layout()
plt.show()

### Character-level and word-level validation error

On the same axis — both unitless and on similar scales. Word error is always ≥ character error (one wrong char makes the whole word wrong).

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(df["epoch"], df["val_error"],      marker="s", color="#b53",
        label="val_error (CER)")
ax.plot(df["epoch"], df["val_word_error"], marker="^", color="#7a4a9a",
        label="val_word_error (WER)")
ax.set_xlabel("epoch")
ax.set_ylabel("error rate")
ax.set_title("Validation error per epoch — character vs word")
ax.legend(loc="upper right")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Best-epoch summary

Match the selection logic in `src/ocr/finetune.py` — best = highest val_accuracy, ties go to the earliest epoch.

In [ ]:
best_idx = df["val_accuracy"].idxmax()  # idxmax returns the FIRST max — matches the script
best = df.loc[best_idx]

print(f"Best epoch by val_accuracy: epoch {int(best['epoch'])}")
print(f"  val_accuracy      = {best['val_accuracy']:.5f}")
print(f"  val_word_accuracy = {best['val_word_accuracy']:.5f}")
print(f"  val_error (CER)   = {best['val_error']:.5f}")
print(f"  val_word_error    = {best['val_word_error']:.5f}")
print(f"  train_loss_epoch  = {best['train_loss_epoch']:.3f}")
print(f"  → checkpoint file: model_{int(best['epoch'])}.mlmodel")